In [11]:
import torch
import json
from torch.utils.data import Dataset
from pathlib import Path

IMSITU_PATH = Path("/kaggle/input/imsitu-annotated")

In [12]:
with open(IMSITU_PATH / "imsitu_annotated_train.json", "r") as f:
    train_data = json.load(f)

with open(IMSITU_PATH / "imsitu_annotated_dev.json", "r") as f:
    val_data = json.load(f)

train_verbs = set([item['verb'] for item in train_data])
val_verbs = set([item['verb'] for item in val_data])

all_verbs = train_verbs.union(val_verbs)
verb_to_idx = {v: i for i, v in enumerate(sorted(list(all_verbs)))} 

In [13]:
from PIL import Image

IMAGES_PATH = Path('/kaggle/input/imsitu-dataset/')

class ImsituDataset(Dataset):
    def __init__(self, source, transform=None, verb2idx=None):
        self.transform = transform

        with open(source, "r") as f:
            self.data = json.load(f)

        # We keep this for reference, but we don't rely on it for model dimension
        self.verbs = sorted(set(i['verb'] for i in self.data))
        self.verb_to_idx = verb2idx

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        
        # Load Image
        img_path = IMAGES_PATH / "of500_images_resized" / item['image_path']
        img = Image.open(img_path).convert("RGB")
        
        if self.transform:
            img = self.transform(img)
            
        # FIX 1: Get the verb string first, then map it to the index
        verb_str = item['verb']
        verb = self.verb_to_idx[verb_str] 
        
        # FIX 2: Convert "M"/"F" strings to 0/1 integers for future leakage steps
        # "M" -> 0, "F" -> 1
        gender_str = item['gender']
        gender = 0 if gender_str == "M" else 1

        return img, verb, gender

# Baseline VGG16

In [14]:
import torch.nn as nn
import torchvision.models as models

class BaselineVGG(nn.Module):
    def __init__(self, num_verbs):
        super(BaselineVGG, self).__init__()
        # Load VGG16 Pretrained
        vgg = models.vgg16(pretrained=True)
        
        # 1. Features: The convolutional blocks (output is 512x7x7)
        self.features = vgg.features
        
        # 2. AvgPool: The adaptive pooling layer (standard VGG uses 7x7 output)
        self.avgpool = vgg.avgpool
        
        # 3. Classifier: The massive dense block (Linear-ReLU-Drop-Linear-ReLU-Drop-Linear)
        self.classifier = vgg.classifier
        
        # Replace the last layer (Index 6) to match your number of verbs
        # Standard VGG last layer is Linear(4096, 1000)
        in_features = self.classifier[6].in_features  # 4096
        self.classifier[6] = nn.Linear(in_features, num_verbs)

    def forward(self, x):
        # Pass through Conv layers
        x = self.features(x)
        x = self.avgpool(x)
        
        # Flatten (VGG requires flattening 512x7x7 -> 25088)
        x = torch.flatten(x, 1)
        
        # Pass through the Dense Classifier Block
        logits = self.classifier(x)
        
        return logits

In [15]:
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm.cli import tqdm  # Specialized for Jupyter
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def train_baseline_model(train_loader, num_verbs, epochs=5):
    model = BaselineVGG(num_verbs).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()

    for epoch in tqdm(range(epochs), desc="Epochs"):
        model.train()
        total_loss = 0

        batch_iter = tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False)

        for images, verbs, _ in batch_iter:
            images, verbs = images.to(device), verbs.to(device)

            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, verbs)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")

    return model

In [16]:
BATCH_SIZE = 32

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = ImsituDataset(IMSITU_PATH / "imsitu_annotated_train.json", transform=train_transform, verb2idx=verb_to_idx)
val_dataset = ImsituDataset(IMSITU_PATH / "imsitu_annotated_dev.json", transform=val_transform, verb2idx=verb_to_idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Training on {len(train_dataset)} images. Validating on {len(val_dataset)} images.")

Training on 33603 images. Validating on 11211 images.


In [17]:
BATCH_SIZE = 32
LR = 1e-4
EPOCHS = 10
SAVE_PATH = "baseline_vgg16_imsitu.pth"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device {device}")

Using device cuda


In [18]:
model = BaselineVGG(num_verbs=len(verb_to_idx)).to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5)

In [19]:
best_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)
    
    for images, verbs, _ in progress_bar:
        images, verbs = images.to(DEVICE), verbs.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, verbs)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
        progress_bar.set_postfix(loss=loss.item())

    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, verbs, _ in tqdm(val_loader, desc="Validating", leave=False):
            images, verbs = images.to(DEVICE), verbs.to(DEVICE)
            
            outputs = model(images)
            loss = criterion(outputs, verbs)
            
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += verbs.size(0)
            correct += (predicted == verbs).sum().item()
            
    avg_val_loss = val_loss / len(val_loader)
    val_acc = 100 * correct / total
    
    scheduler.step(avg_val_loss)

    print(f"Epoch [{epoch+1}/{EPOCHS}] | "
          f"Train Loss: {running_loss/len(train_loader):.4f} | "
          f"Val Loss: {avg_val_loss:.4f} | "
          f"Val Acc: {val_acc:.2f}%")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"--> Best Model Saved (Acc: {best_acc:.2f}%)")

print("Training Complete.")

Epoch [1/10] | Train Loss: 5.4480 | Val Loss: 4.9219 | Val Acc: 7.54%
--> Best Model Saved (Acc: 7.54%)


Epoch [2/10] | Train Loss: 4.5813 | Val Loss: 4.4848 | Val Acc: 12.01%
--> Best Model Saved (Acc: 12.01%)


Epoch [3/10] | Train Loss: 4.0554 | Val Loss: 4.1415 | Val Acc: 15.89%
--> Best Model Saved (Acc: 15.89%)


Epoch [4/10] | Train Loss: 3.6251 | Val Loss: 3.9112 | Val Acc: 19.69%
--> Best Model Saved (Acc: 19.69%)


Epoch [5/10] | Train Loss: 3.2530 | Val Loss: 3.9128 | Val Acc: 19.54%


Epoch [6/10] | Train Loss: 2.8759 | Val Loss: 3.9319 | Val Acc: 21.68%
--> Best Model Saved (Acc: 21.68%)


Epoch [7/10] | Train Loss: 2.5495 | Val Loss: 3.8575 | Val Acc: 22.14%
--> Best Model Saved (Acc: 22.14%)


Epoch [8/10] | Train Loss: 2.2297 | Val Loss: 3.9318 | Val Acc: 22.67%
--> Best Model Saved (Acc: 22.67%)


Epoch [9/10] | Train Loss: 1.9347 | Val Loss: 4.1004 | Val Acc: 22.80%
--> Best Model Saved (Acc: 22.80%)


Epoch [10/10] | Train Loss: 1.6746 | Val Loss: 4.1912 | Val Acc: 22.95%
--> Best Model Saved (Acc: 22.95%)
Training Complete.


In [20]:
!pip install pyuploadcare -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 367.8/367.8 kB 13.6 MB/s eta 0:00:00


In [21]:
from pyuploadcare import Uploadcare, File

uploadcare = Uploadcare(public_key='43189ed23a5312d4ede8', secret_key='598681585144a03347f7')
with open('baseline_vgg16_imsitu.pth', 'rb') as file_object:
    ucare_file = uploadcare.upload(file_object)

## Dataset Leakage and Model Leakage of Baseline VGG16

In [22]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from tqdm.notebook import tqdm

class Attacker(nn.Module):
    def __init__(self, input_dim, hidden_dim=300):
        super(Attacker, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.LeakyReLU(),
            
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.LeakyReLU(),
            
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.LeakyReLU(),
            
            nn.Linear(hidden_dim, 2)  # Binary Gender (0 or 1)
        )

    def forward(self, x):
        return self.net(x)

In [38]:
def get_dataset_features(loader, num_verbs, device):
    """
    Extracts Ground Truth features (One-Hot Verbs) and Gender labels.
    """
    all_features = []
    all_genders = []
    
    for _, verbs, genders in tqdm(loader, desc="Extracting Dataset Features", leave=False):
        # Convert integer verbs to One-Hot float tensors
        one_hot = F.one_hot(verbs, num_classes=num_verbs).float()
        
        all_features.append(one_hot)
        all_genders.append(genders)
        
    return torch.cat(all_features).to(device), torch.cat(all_genders).to(device)

def get_model_features(baseline_model, loader, device):
    """
    Extracts Model Logits (features) and Gender labels.
    """
    baseline_model.eval()
    all_logits = []
    all_genders = []
    
    with torch.no_grad():
        for images, _, genders in tqdm(loader, desc="Extracting Model Logits", leave=False):
            images = images.to(device)
            
            # Forward pass to get logits (before Softmax)
            outputs = baseline_model(images)

            if isinstance(outputs, tuple):
                logits = outputs[0]
            else:
                logits = outputs
            
            all_logits.append(logits)
            all_genders.append(genders)
            
    return torch.cat(all_logits).to(device), torch.cat(all_genders).to(device)

In [42]:
from tqdm.cli import tqdm

def train_attacker(X_train, y_train, X_test, y_test, device, name="Attacker"):
    # Create datasets
    train_ds = TensorDataset(X_train, y_train)
    test_ds = TensorDataset(X_test, y_test)
    
    train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
    test_dl = DataLoader(test_ds, batch_size=128, shuffle=False)
    
    # Initialize attacker model
    input_dim = X_train.shape[1]
    model = Attacker(input_dim=input_dim).to(device)
    
    optimizer = optim.Adam(model.parameters(), lr=1e-4)  # Standard LR for MLP
    criterion = nn.CrossEntropyLoss()
    
    best_acc = 0.0
    epochs = 20  # MLPs converge quickly
    
    # ---- Epoch loop with tqdm ----
    for epoch in tqdm(range(epochs), desc=f"{name} Training"):
        model.train()

        # ---- Batch loop with tqdm ----
        train_iter = tqdm(train_dl, desc=f"Epoch {epoch+1}", leave=False)

        for features, targets in train_iter:
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
        
        # ---- Evaluation ----
        model.eval()
        correct = 0
        total = 0
        
        with torch.no_grad():
            for features, targets in test_dl:
                outputs = model(features)
                _, predicted = torch.max(outputs.data, 1)
                total += targets.size(0)
                correct += (predicted == targets).sum().item()
        
        acc = 100 * correct / total
        if acc > best_acc:
            best_acc = acc
    
    print(f"{name} Results | Best Accuracy: {best_acc:.2f}%")
    return best_acc

In [25]:
num_verbs = len(verb_to_idx)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 1. CALCULATE DATASET LEAKAGE (Lambda_D) ---
print("\n--- Calculating Dataset Leakage (Lambda_D) ---")
# Extract One-Hot Vectors
d_X_train, d_y_train = get_dataset_features(train_loader, num_verbs, DEVICE)
d_X_val, d_y_val = get_dataset_features(val_loader, num_verbs, DEVICE)

# Train Attacker on Ground Truth
lambda_d = train_attacker(d_X_train, d_y_train, d_X_val, d_y_val, DEVICE, name="Dataset Leakage")


--- Calculating Dataset Leakage (Lambda_D) ---


Dataset Leakage Training: 100%|██████████| 20/20 [00:19<00:00,  1.05it/s]

Dataset Leakage Results | Best Accuracy: 68.24%


In [26]:
num_verbs = len(verb_to_idx)

# A. Instantiate the empty architecture
loaded_model = BaselineVGG(num_verbs=num_verbs).to(DEVICE)

# B. Load the weights
loaded_model.load_state_dict(torch.load("baseline_vgg16_imsitu.pth", map_location=DEVICE))

# C. Set to Evaluation Mode (Critical for consistent feature extraction)
loaded_model.eval()

print("\n--- Calculating Model Leakage (Lambda_M) ---")
# Extract Model Logits
m_X_train, m_y_train = get_model_features(loaded_model, train_loader, DEVICE)
m_X_val, m_y_val = get_model_features(loaded_model, val_loader, DEVICE)

# Train Attacker on Model Predictions
lambda_m = train_attacker(m_X_train, m_y_train, m_X_val, m_y_val, DEVICE, name="Model Leakage")


--- Calculating Model Leakage (Lambda_M) ---


Model Leakage Training: 100%|██████████| 20/20 [00:19<00:00,  1.03it/s]

Model Leakage Results | Best Accuracy: 71.22%


In [27]:
amplification = lambda_m - lambda_d

print("\n" + "="*40)
print(f"Dataset Leakage (Lambda_D): {lambda_d:.2f}%")
print(f"Model Leakage   (Lambda_M): {lambda_m:.2f}%")
print(f"Bias Amplification (Delta): {amplification:.2f}%")
print("="*40)


Dataset Leakage (Lambda_D): 68.24%
Model Leakage   (Lambda_M): 71.22%
Bias Amplification (Delta): 2.99%


# VGG16 `adv@conv4`

In [28]:
import torch
import torch.nn as nn
import torchvision.models as models
from torch.autograd import Function

# Ensure GradientReversal is defined in your notebook
class GradientReversal(Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        output = grad_output.neg() * ctx.alpha
        return output, None

class DebiasedVGGConv4(nn.Module):
    def __init__(self, num_verbs, num_genders=2):
        super(DebiasedVGGConv4, self).__init__()
        
        # Load Pretrained VGG16
        vgg = models.vgg16(pretrained=True)
        
        # --- 1. SPLIT THE BACKBONE ---
        
        # Part 1: Input -> ... -> Block 4 MaxPool (Index 0 to 23)
        # Output Shape: [Batch, 512, 14, 14]
        self.features_part1 = vgg.features[:24]
        
        # Part 2: Block 5 (Index 24 to 30) -> Output [Batch, 512, 7, 7]
        self.features_part2 = vgg.features[24:]
        
        # Pooling & Classifier (Standard VGG)
        self.avgpool = vgg.avgpool
        self.classifier = vgg.classifier
        
        # Replace final layer for verbs
        in_features = self.classifier[6].in_features
        self.classifier[6] = nn.Linear(in_features, num_verbs)

        # --- 2. SPATIAL ADVERSARY ---
        # Input is [Batch, 512, 14, 14]
        self.adversary_conv = nn.Sequential(
            # Conv 1: 512 -> 256 (Adjusted for VGG channel depth)
            nn.Conv2d(512, 256, kernel_size=1), 
            nn.BatchNorm2d(256),
            nn.LeakyReLU(),
            
            # Conv 2: 256 -> 256
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(),
            
            # Conv 3: 256 -> 256
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(),
            
            # Flatten spatial dims
            nn.AdaptiveAvgPool2d((1, 1))
        )
        
        self.adversary_fc = nn.Sequential(
            nn.Flatten(),
            # Linear Layers (Same as before)
            nn.Linear(256, 256), nn.BatchNorm1d(256), nn.LeakyReLU(),
            nn.Linear(256, 256), nn.BatchNorm1d(256), nn.LeakyReLU(),
            nn.Linear(256, 256), nn.BatchNorm1d(256), nn.LeakyReLU(),
            # Output
            nn.Linear(256, num_genders)
        )

    def forward(self, x, alpha=1.0):
        # 1. Extract mid-level features (Conv4)
        # Output: [Batch, 512, 14, 14]
        f_conv4 = self.features_part1(x)
        
        # --- PATH A: Main Task (Verb Recognition) ---
        # Finish the VGG features (Block 5)
        f_final = self.features_part2(f_conv4)
        f_final = self.avgpool(f_final)
        f_final = torch.flatten(f_final, 1)
        verb_pred = self.classifier(f_final)
        
        # --- PATH B: Adversary (Gender Prediction) ---
        # Apply GRL to the spatial features
        reversed_f = GradientReversal.apply(f_conv4, alpha)
        
        # Process with spatial adversary
        adv_feat = self.adversary_conv(reversed_f)
        gender_pred = self.adversary_fc(adv_feat)

        return verb_pred, gender_pred

In [29]:
BATCH_SIZE = 32
LR = 1e-4
EPOCHS = 10
SAVE_PATH = "debaised_vgg16_adv@conv4.pth"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device {DEVICE}")

Using device cuda


In [30]:
import numpy as np

def get_alpha(current_step, total_steps):
    """
    Calculates lambda (alpha) using the schedule from Ganin et al. (DANN),
    which is standard for this type of GRL training.
    """
    p = float(current_step) / total_steps
    return 2. / (1. + np.exp(-10 * p)) - 1

In [31]:
num_verbs = len(verb_to_idx)
model = DebiasedVGGConv4(num_verbs=num_verbs).to(DEVICE)

# Optimizers
optimizer = optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

# Mixed Precision
scaler = torch.cuda.amp.GradScaler()

/tmp/ipykernel_47/2940340716.py:9: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [32]:
best_acc = 0.0
total_steps = EPOCHS * len(train_loader)
current_step = 0

print("Starting Adversarial Training (adv@conv4)...")

for epoch in range(EPOCHS):
    model.train()
    
    total_verb_loss = 0
    total_gender_loss = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)
    
    for images, verbs, genders in pbar:
        images, verbs, genders = images.to(DEVICE), verbs.to(DEVICE), genders.to(DEVICE)
        
        # 1. Calculate Alpha (Lambda) for this step
        alpha = get_alpha(current_step, total_steps)
        current_step += 1
        
        optimizer.zero_grad()
        
        with torch.cuda.amp.autocast():
            # 2. Forward Pass
            # We pass alpha so the GRL knows how much to reverse gradients
            verb_pred, gender_pred = model(images, alpha=alpha)
            
            # 3. Calculate Losses
            loss_verb = criterion(verb_pred, verbs)
            loss_gender = criterion(gender_pred, genders)
            
            # 4. Total Loss
            # CRITICAL NOTE: We ADD the losses. 
            # The GRL layer inside the model will automatically FLIP the sign 
            # of the gradient coming from 'gender_pred' during backward().
            # So: Minimize Verb Error AND (Minimize -Gender Error) -> Maximize Gender Error
            total_loss = loss_verb + loss_gender
            
        # 5. Backward & Step
        scaler.scale(total_loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_verb_loss += loss_verb.item()
        total_gender_loss += loss_gender.item()
        
        pbar.set_postfix(v_loss=loss_verb.item(), g_loss=loss_gender.item(), alpha=f"{alpha:.2f}")

    # --- VALIDATION (Check Verb Accuracy) ---
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, verbs, _ in tqdm(val_loader, desc="Validating", leave=False):
            images, verbs = images.to(DEVICE), verbs.to(DEVICE)
            
            # For validation, we don't care about the adversary output or alpha
            verb_pred, _ = model(images, alpha=0.0) 
            
            _, predicted = torch.max(verb_pred.data, 1)
            total += verbs.size(0)
            correct += (predicted == verbs).sum().item()
            
    val_acc = 100 * correct / total
    
    print(f"Epoch {epoch+1} Results:")
    print(f"  Verb Loss: {total_verb_loss/len(train_loader):.4f}")
    print(f"  Gender Loss: {total_gender_loss/len(train_loader):.4f} (Adversary Performance)")
    print(f"  Val Accuracy: {val_acc:.2f}%")
    
    # Save Best
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"  --> Best Model Saved")

print("Adversarial Training Complete.")

Starting Adversarial Training (adv@conv4)...


Epoch 1/10:   0%|          | 0/1051 [00:00<?, ?it/s]/tmp/ipykernel_47/1824844427.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 1 Results:
  Verb Loss: 5.6578
  Gender Loss: 0.6618 (Adversary Performance)
  Val Accuracy: 2.80%
  --> Best Model Saved


Epoch 2 Results:
  Verb Loss: 5.0981
  Gender Loss: 0.6532 (Adversary Performance)
  Val Accuracy: 5.97%
  --> Best Model Saved


Epoch 3 Results:
  Verb Loss: 4.6406
  Gender Loss: 0.6422 (Adversary Performance)
  Val Accuracy: 10.27%
  --> Best Model Saved


...

Epoch 6 Results:
  Verb Loss: 3.6296
  Gender Loss: 0.6217 (Adversary Performance)
  Val Accuracy: 16.97%
  --> Best Model Saved


Epoch 7 Results:
  Verb Loss: 3.3379
  Gender Loss: 0.6155 (Adversary Performance)
  Val Accuracy: 17.82%
  --> Best Model Saved


Epoch 8 Results:
  Verb Loss: 3.0540
  Gender Loss: 0.6107 (Adversary Performance)
  Val Accuracy: 18.49%
  --> Best Model Saved


Epoch 9 Results:
  Verb Loss: 2.7713
  Gender Loss: 0.6026 (Adversary Performance)
  Val Accuracy: 19.19%
  --> Best Model Saved


Epoch 10 Results:
  Verb Loss: 2.4867
  Gender Loss: 0.5970 (Adversary Performance)
  Val Accuracy: 19.48%
  --> Best Model Saved
Adversarial Training Complete.


In [33]:
from pyuploadcare import Uploadcare, File

uploadcare = Uploadcare(public_key='43189ed23a5312d4ede8', secret_key='598681585144a03347f7')
with open('debaised_vgg16_adv@conv4.pth', 'rb') as file_object:
    ucare_file = uploadcare.upload(file_object)

## Dataset and Model Leakage of VGG16 `adv@conv4`

In [39]:
num_verbs = len(verb_to_idx)

# A. Instantiate the empty architecture
loaded_model = DebiasedVGGConv4(num_verbs=num_verbs).to(DEVICE)

# B. Load the weights
loaded_model.load_state_dict(torch.load("debaised_vgg16_adv@conv4.pth", map_location=DEVICE))

# C. Set to Evaluation Mode (Critical for consistent feature extraction)
loaded_model.eval()

print("\n--- Calculating Model Leakage (Lambda_M) ---")
# Extract Model Logits
m_X_train, m_y_train = get_model_features(loaded_model, train_loader, DEVICE)
m_X_val, m_y_val = get_model_features(loaded_model, val_loader, DEVICE)

# Train Attacker on Model Predictions
lambda_m = train_attacker(m_X_train, m_y_train, m_X_val, m_y_val, DEVICE, name="Model Leakage")


--- Calculating Model Leakage (Lambda_M) ---


Model Leakage Training: 100%|██████████| 20/20 [00:19<00:00,  1.03it/s]

Model Leakage Results | Best Accuracy: 64.14%


In [40]:
amplification = lambda_m - lambda_d

print("\n" + "="*40)
print(f"Dataset Leakage (Lambda_D): {lambda_d:.2f}%")
print(f"Model Leakage   (Lambda_M): {lambda_m:.2f}%")
print(f"Bias Amplification (Delta): {amplification:.2f}%")
print("="*40)


Dataset Leakage (Lambda_D): 68.24%
Model Leakage   (Lambda_M): 64.14%
Bias Amplification (Delta): -4.09%


# VGG16 `adv@conv5`

In [43]:
import torch
import torch.nn as nn
import torchvision.models as models
from torch.autograd import Function

# Ensure GradientReversal is defined in your notebook
class GradientReversal(Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        output = grad_output.neg() * ctx.alpha
        return output, None

class DebiasedVGGConv5(nn.Module):
    def __init__(self, num_verbs, num_genders=2):
        super(DebiasedVGGConv5, self).__init__()
        
        # Load VGG16
        vgg = models.vgg16(pretrained=True)
        
        # 1. Feature Extractor
        # Includes all Conv layers + AdaptiveAvgPool
        # Output is [Batch, 512, 7, 7]
        self.features = vgg.features
        self.avgpool = vgg.avgpool
        
        # 2. Main Task Classifier (The standard VGG Dense Block)
        # Input: 25088 (512*7*7) -> 4096 -> 4096 -> num_verbs
        self.classifier = vgg.classifier
        
        # Modify the final layer for our task
        in_features = self.classifier[6].in_features  # 4096
        self.classifier[6] = nn.Linear(in_features, num_verbs)

        # 3. Adversary / Critic (Gender Prediction)
        # Input: Vectorized feature map (512*7*7 = 25088)
        # Note: 25088 input makes this layer quite heavy
        self.adversary = nn.Sequential(
            nn.Linear(512 * 7 * 7, 300),
            nn.BatchNorm1d(300),
            nn.LeakyReLU(),
            
            nn.Linear(300, 300),
            nn.BatchNorm1d(300),
            nn.LeakyReLU(),
            
            nn.Linear(300, 300),
            nn.BatchNorm1d(300),
            nn.LeakyReLU(),
            
            nn.Linear(300, num_genders)
        )

    def forward(self, x, alpha=1.0):
        # 1. Extract Features
        x = self.features(x)
        x = self.avgpool(x)
        
        # Flatten: [Batch, 512, 7, 7] -> [Batch, 25088]
        f_vector = torch.flatten(x, 1)

        # --- PATH A: Main Task ---
        verb_pred = self.classifier(f_vector)

        # --- PATH B: Adversary ---
        # Apply GRL to the flattened vector
        reversed_f = GradientReversal.apply(f_vector, alpha)
        gender_pred = self.adversary(reversed_f)

        return verb_pred, gender_pred

In [44]:
BATCH_SIZE = 32
LR = 1e-4
EPOCHS = 10
SAVE_PATH = "debaised_vgg16_adv@conv5.pth"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device {DEVICE}")

Using device cuda


In [45]:
num_verbs = len(verb_to_idx)
model = DebiasedVGGConv5(num_verbs=num_verbs).to(DEVICE)

# Optimizers
optimizer = optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

# Mixed Precision
scaler = torch.cuda.amp.GradScaler()

/tmp/ipykernel_47/2814511581.py:9: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [46]:
best_acc = 0.0
total_steps = EPOCHS * len(train_loader)
current_step = 0

print("Starting Adversarial Training (adv@conv4)...")

for epoch in range(EPOCHS):
    model.train()
    
    total_verb_loss = 0
    total_gender_loss = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)
    
    for images, verbs, genders in pbar:
        images, verbs, genders = images.to(DEVICE), verbs.to(DEVICE), genders.to(DEVICE)
        
        # 1. Calculate Alpha (Lambda) for this step
        alpha = get_alpha(current_step, total_steps)
        current_step += 1
        
        optimizer.zero_grad()
        
        with torch.cuda.amp.autocast():
            # 2. Forward Pass
            # We pass alpha so the GRL knows how much to reverse gradients
            verb_pred, gender_pred = model(images, alpha=alpha)
            
            # 3. Calculate Losses
            loss_verb = criterion(verb_pred, verbs)
            loss_gender = criterion(gender_pred, genders)
            
            # 4. Total Loss
            # CRITICAL NOTE: We ADD the losses. 
            # The GRL layer inside the model will automatically FLIP the sign 
            # of the gradient coming from 'gender_pred' during backward().
            # So: Minimize Verb Error AND (Minimize -Gender Error) -> Maximize Gender Error
            total_loss = loss_verb + loss_gender
            
        # 5. Backward & Step
        scaler.scale(total_loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_verb_loss += loss_verb.item()
        total_gender_loss += loss_gender.item()
        
        pbar.set_postfix(v_loss=loss_verb.item(), g_loss=loss_gender.item(), alpha=f"{alpha:.2f}")

    # --- VALIDATION (Check Verb Accuracy) ---
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, verbs, _ in tqdm(val_loader, desc="Validating", leave=False):
            images, verbs = images.to(DEVICE), verbs.to(DEVICE)
            
            # For validation, we don't care about the adversary output or alpha
            verb_pred, _ = model(images, alpha=0.0) 
            
            _, predicted = torch.max(verb_pred.data, 1)
            total += verbs.size(0)
            correct += (predicted == verbs).sum().item()
            
    val_acc = 100 * correct / total
    
    print(f"Epoch {epoch+1} Results:")
    print(f"  Verb Loss: {total_verb_loss/len(train_loader):.4f}")
    print(f"  Gender Loss: {total_gender_loss/len(train_loader):.4f} (Adversary Performance)")
    print(f"  Val Accuracy: {val_acc:.2f}%")
    
    # Save Best
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"  --> Best Model Saved")

print("Adversarial Training Complete.")

Starting Adversarial Training (adv@conv4)...


Epoch 1/10:   0%|          | 0/1051 [00:00<?, ?it/s]/tmp/ipykernel_47/1824844427.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 1 Results:
  Verb Loss: 5.4630
  Gender Loss: 0.6519 (Adversary Performance)
  Val Accuracy: 6.53%
  --> Best Model Saved


Epoch 2 Results:
  Verb Loss: 4.6350
  Gender Loss: 0.6342 (Adversary Performance)
  Val Accuracy: 11.68%
  --> Best Model Saved


Epoch 3/10:  85%|████████▍ | 891/1051 [03:55<00:42,  3.80it/s, alpha=0.89, g_loss=0.649, v_loss=3.8] Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f222bf58720>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py", line 1618, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py", line 1601, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.11/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Epoch 3/10:  85%|████████▌ | 894/1051 [03:56<00:41,  3.78it/s, alpha=0.89, g_loss=0.607, v_loss=3.85]Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f222bf58720>
Traceback (most recent call last):
  File "/usr/local

...

Epoch 6 Results:
  Verb Loss: 3.0077
  Gender Loss: 0.5850 (Adversary Performance)
  Val Accuracy: 21.18%
  --> Best Model Saved


Epoch 7 Results:
  Verb Loss: 2.6987
  Gender Loss: 0.5646 (Adversary Performance)
  Val Accuracy: 21.54%
  --> Best Model Saved


Epoch 8 Results:
  Verb Loss: 2.4147
  Gender Loss: 0.5423 (Adversary Performance)
  Val Accuracy: 22.27%
  --> Best Model Saved


Epoch 9 Results:
  Verb Loss: 2.1517
  Gender Loss: 0.5257 (Adversary Performance)
  Val Accuracy: 21.21%


Epoch 10 Results:
  Verb Loss: 1.9197
  Gender Loss: 0.5015 (Adversary Performance)
  Val Accuracy: 21.56%
Adversarial Training Complete.


## Dataset and Model Leakage of VGG16 `adv@conv5`

In [47]:
num_verbs = len(verb_to_idx)

# A. Instantiate the empty architecture
loaded_model = DebiasedVGGConv5(num_verbs=num_verbs).to(DEVICE)

# B. Load the weights
loaded_model.load_state_dict(torch.load("debaised_vgg16_adv@conv5.pth", map_location=DEVICE))

# C. Set to Evaluation Mode (Critical for consistent feature extraction)
loaded_model.eval()

print("\n--- Calculating Model Leakage (Lambda_M) ---")
# Extract Model Logits
m_X_train, m_y_train = get_model_features(loaded_model, train_loader, DEVICE)
m_X_val, m_y_val = get_model_features(loaded_model, val_loader, DEVICE)

# Train Attacker on Model Predictions
lambda_m = train_attacker(m_X_train, m_y_train, m_X_val, m_y_val, DEVICE, name="Model Leakage")


--- Calculating Model Leakage (Lambda_M) ---


Model Leakage Training: 100%|██████████| 20/20 [00:19<00:00,  1.02it/s]

Model Leakage Results | Best Accuracy: 64.10%


In [48]:
amplification = lambda_m - lambda_d

print("\n" + "="*40)
print(f"Dataset Leakage (Lambda_D): {lambda_d:.2f}%")
print(f"Model Leakage   (Lambda_M): {lambda_m:.2f}%")
print(f"Bias Amplification (Delta): {amplification:.2f}%")
print("="*40)


Dataset Leakage (Lambda_D): 68.24%
Model Leakage   (Lambda_M): 64.10%
Bias Amplification (Delta): -4.14%
